[개선 프로젝트 주제] "생물학적 계층 구조 및 유전적 유사성(KIPAN-KIRC 등)을 활용한 26개 암종 멀티클래스 분류 모델 성능 개선"

배경 및 한계점: 4,384개의 고차원 유전자 변이 데이터와 극희소(Sparse, ~0.81%) 특성을 가진 26개 암종 데이터 학습 시, 단순 플랫(Flat) 분류 방식은 클래스 간 경계 모호성과 과적합으로 인해 성능 한계에 직면함.   

핵심 전처리 및 모델링 전략
1. 유전적 계층화 및 그룹화: KIPAN-KIRC 관계와 같이 생물학적/통계적으로 유사한 암종군 간의 공통 변이 패턴을 선행 학습.
2. 계층적/멀티태스크 구조 도입: 상위 암종 그룹 예측과 26개 세부 암종 예측을 연계하여 데이터 희소성이 높은 클래스의 학습 보완.
3. 희소 데이터 최적화: 0.81%의 낮은 Mutation Ratio 속에서 유의미한 패턴을 추출할 수 있도록 피처 스페이스 재구성.   
4. 기대 효과: 유사 암종 간 오분류(False Positive/Negative) 감소 및 소수 클래스(예: DLBC 등)에 대한 일반화 성능 향상.

[Raw Data: 4,384 Genes, 6,201 Samples, 26 Subclasses]
       ↓
Step 1: 메타 암종(예: KIPAN)과 세부 암종(예: KIRC) 간 계층 레이블(Hierarchical Label) 매핑
       ↓
Step 2: 유전자 변이 프로필 기반 유사도 행렬 생성 및 암종별 클러스터링(Group)
       ↓
Step 3: 희소성(0.81%) 대응 및 유전자 피처 가공 (차원 축소 / 공발생 패턴 피처)
       ↓
Step 4: 계층적 멀티태스크(Multi-task) 학습용 PyTorch Dataset & Model 구성

In [ ]:
import numpy    as np
import pandas   as pd

from scipy.sparse       import csr_matrix
from sklearn.cluster    import AgglomerativeClustering
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset


# =============================================================
# [Step 1] 데이터 로드 및 유전자 변이 이진화
# =============================================================

# 1. 데이터 로드
train = pd.read_csv("./sample/train.csv")
test = pd.read_csv("./sample/test.csv")

print(f"Train 데이터 크기: {train.shape}")
print(f"Test 데이터 크기: {test.shape}")

# 2. 유전자 피처 컬럼 추출
# ID와 정답 및 파생 레이블은 제외
exclude_cols = [
    'ID',
    'SUBCLASS',
    'SUPER_CLASS',
    'sub_label',
    'super_label'
]

gene_cols = [
    col for col in train.columns
    if col not in exclude_cols
]

# Test 컬럼 정합성 확인
assert set(gene_cols).issubset(set(test.columns)), \
    "Test 데이터에 Train 유전자 컬럼이 존재하지 않습니다."

print(f"추출된 유전자 피처 개수: {len(gene_cols)}개")

# 3. 원본 결측치 확인
print(
    f"Train 결측 칸: "
    f"{int(train[gene_cols].isnull().sum().sum())}"
)

print(
    f"Test 결측 칸: "
    f"{int(test[gene_cols].isnull().sum().sum())}"
)

# 4. WT 및 결측치 처리
# 결측치와 WT는 0, 그 외 변이는 1
for col in gene_cols:

    train[col] = train[col].fillna('WT')
    test[col] = test[col].fillna('WT')

    train[col] = (
        train[col]
        .astype(str)
        .str.strip()
        .ne('WT')
        .astype(np.int8)
    )

    test[col] = (
        test[col]
        .astype(str)
        .str.strip()
        .ne('WT')
        .astype(np.int8)
    )

print("\nWT 및 결측치 이진화 완료!")

print(
    f"Train 결측 칸: "
    f"{int(train[gene_cols].isnull().sum().sum())}"
)

print(
    f"Test 결측 칸: "
    f"{int(test[gene_cols].isnull().sum().sum())}"
)

Train 데이터 크기: (6201, 4386)
Test 데이터 크기: (2546, 4385)
추출된 유전자 피처 개수: 4384개
Train 결측 칸: 0
Test 결측 칸: 237

WT 및 결측치 이진화 완료!
Train 결측 칸: 0
Test 결측 칸: 0


In [2]:
# -------------------------------------------------------------
# [Step 2] 계층적 레이블(Super Class) 매핑 정의 (연구 질문 1, 2 대응)
# -------------------------------------------------------------
# 예: KIPAN 내부에 KIRC, KIRP 등이 포함되는 구조나, 유사한 장기/계열별 그룹 정의
def get_super_class_mapping(subclass):

    # 1. 신장암 계열
    if subclass in ["KIRC", "KIRP", "KICH", "KIPAN"]:
        return "Kidney_Group"

    # 2. 폐암 계열
    elif subclass in ["LUAD", "LUSC"]:
        return "Lung_Group"

    # 3. 소화기 계열
    elif subclass in ["STES", "COAD", "LIHC", "PAAD"]:
        return "GI_Digestive_Group"

    # 4. 뇌 및 신경계
    elif subclass in ["GBMLGG", "LGG"]:
        return "CNS_Brain_Group"

    # 5. 여성 생식기 계열
    elif subclass in ["BRCA", "CESC", "UCEC", "OV"]:
        return "Female_Reproductive_Group"

    # 6. 남성 생식기 및 비뇨기 계열
    elif subclass in ["PRAD", "BLCA", "TGCT"]:
        return "Male_Genitourinary_Group"

    # 7. 두경부 계열
    elif subclass in ["HNSC"]:
        return "Head_Neck_Group"

    # 8. 내분비 및 희귀 암종
    elif subclass in ["THCA", "THYM", "ACC", "PCPG"]:
        return "Endocrine_Rare_Group"

    # 9. 피부 및 육종
    elif subclass in ["SKCM", "SARC"]:
        return "Skin_Sarcoma_Group"

    # 10. 조혈 및 림프계
    elif subclass in ["LAML", "DLBC"]:
        return "Hematopoietic_Group"

    else:
        return "Other_Group"


# 상위 그룹 레이블 생성
train["SUPER_CLASS"] = (
    train["SUBCLASS"]
    .apply(get_super_class_mapping)
)

# Label Encoding
sub_encoder = LabelEncoder()
super_encoder = LabelEncoder()

train["sub_label"] = (
    sub_encoder.fit_transform(train["SUBCLASS"])
)

train["super_label"] = (
    super_encoder.fit_transform(train["SUPER_CLASS"])
)

num_sub_classes = len(sub_encoder.classes_)
num_super_classes = len(super_encoder.classes_)

print("\n[계층적 레이블 생성 결과]")
print(f"세부 암종(Subclass): {num_sub_classes}개")
print(f"상위 그룹(Super Class): {num_super_classes}개")

print("\n[상위 그룹별 암종 분포]")
print(
    train.groupby("SUPER_CLASS")["SUBCLASS"]
    .value_counts()
)


[계층적 레이블 생성 결과]
세부 암종(Subclass): 26개
상위 그룹(Super Class): 10개

[상위 그룹별 암종 분포]
SUPER_CLASS                SUBCLASS
CNS_Brain_Group            GBMLGG      461
                           LGG         229
Endocrine_Rare_Group       THCA        324
                           PCPG        147
                           THYM         98
                           ACC          72
Female_Reproductive_Group  BRCA        786
                           OV          253
                           UCEC        198
                           CESC        155
GI_Digestive_Group         STES        379
                           COAD        223
                           LIHC        158
                           PAAD        120
Head_Neck_Group            HNSC        223
Hematopoietic_Group        LAML        158
                           DLBC         38
Kidney_Group               KIPAN       515
                           KIRC        334
Lung_Group                 LUAD        184
                          

/var/folders/gm/wx_04p912ys05gn0dyy5sw3h0000gn/T/ipykernel_6152/3889776000.py:52: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train["SUPER_CLASS"] = (
/var/folders/gm/wx_04p912ys05gn0dyy5sw3h0000gn/T/ipykernel_6152/3889776000.py:61: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train["sub_label"] = (
/var/folders/gm/wx_04p912ys05gn0dyy5sw3h0000gn/T/ipykernel_6152/3889776000.py:65: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  C

In [3]:
# =============================================================
# [Step 3] 데이터 분리 및 누수 없는 피처 정제
# =============================================================

print("\n고차원 유전자 변이 데이터 정제 시작...")

X_train_raw = train[gene_cols].astype(np.uint8).copy()
X_test_raw = test[gene_cols].astype(np.uint8).copy()

# 이진화 상태 확인
assert X_train_raw.isin([0, 1]).all().all()
assert X_test_raw.isin([0, 1]).all().all()

y_sub = train["sub_label"]
y_super = train["super_label"]

# Train / Validation 분리
train_idx, val_idx = train_test_split(
    np.arange(len(train)),
    test_size=0.2,
    random_state=42,
    stratify=y_sub
)

X_train_split = X_train_raw.iloc[train_idx].copy()
X_val_split = X_train_raw.iloc[val_idx].copy()

y_sub_train = y_sub.iloc[train_idx].copy()
y_sub_val = y_sub.iloc[val_idx].copy()

y_super_train = y_super.iloc[train_idx].copy()
y_super_val = y_super.iloc[val_idx].copy()

# 학습 데이터만으로 Active Gene 선택
active_genes = X_train_split.columns[
    X_train_split.sum(axis=0) > 0
]

X_train_filtered = X_train_split[active_genes].copy()
X_val_filtered = X_val_split[active_genes].copy()
X_test_filtered = X_test_raw[active_genes].copy()

assert list(X_train_filtered.columns) == list(X_val_filtered.columns)
assert list(X_train_filtered.columns) == list(X_test_filtered.columns)

X_train_gene = X_train_filtered.to_numpy(dtype=np.float32)
X_val_gene = X_val_filtered.to_numpy(dtype=np.float32)
X_test_gene = X_test_filtered.to_numpy(dtype=np.float32)

print("\n모델 입력 데이터 변환 완료!")


고차원 유전자 변이 데이터 정제 시작...

모델 입력 데이터 변환 완료!


In [4]:
def analyze_subclass_similarity(
    X_data,
    y_subclass,
    n_clusters=5
):

    # 1. 입력 데이터 DataFrame 변환
    temp_df = pd.DataFrame(X_data)

    # 2. SUBCLASS 레이블 결합
    subclass_series = pd.Series(
        np.asarray(y_subclass),
        index=temp_df.index,
        name="SUBCLASS"
    )

    temp_df["SUBCLASS"] = subclass_series

    # 3. 암종별 평균 유전자 변이 패턴 계산
    centroids = (
        temp_df
        .groupby("SUBCLASS")
        .mean()
    )

    # 4. 클러스터 개수 유효성 확인
    if n_clusters < 2 or n_clusters > len(centroids):
        raise ValueError(
            "n_clusters는 2 이상, 암종 개수 이하이어야 합니다."
        )

    # 5. 계층적 클러스터링
    clustering = AgglomerativeClustering(
        n_clusters=n_clusters,
        metric="euclidean",
        linkage="ward"
    )

    cluster_ids = clustering.fit_predict(centroids)

    # 6. 결과 정리
    similarity_report = pd.DataFrame({
        "Subclass": centroids.index,
        "Cluster_ID": cluster_ids
    })

    # 7. 암종별 클러스터 결과 반환
    return similarity_report, centroids


# Train 데이터만 사용하여 유사성 분석
# Validation 데이터는 클러스터링 과정에 사용하지 않음
similarity_df, subclass_centroids = (
    analyze_subclass_similarity(
        X_train_gene,
        train["SUBCLASS"].iloc[train_idx],
        n_clusters=5
    )
)

print("\n[암종별 유전적 유사성 클러스터링 결과]")
print(
    similarity_df.sort_values("Cluster_ID")
    .to_string(index=False)
)

print("\n[클러스터별 암종 구성]")
for cluster_id, group in similarity_df.groupby("Cluster_ID"):

    print(
        f"\nCluster {cluster_id}: "
        f"{group['Subclass'].tolist()}"
    )

print("\n[클러스터링 분석 완료]")


[암종별 유전적 유사성 클러스터링 결과]
Subclass  Cluster_ID
     ACC           0
    THCA           0
    TGCT           0
    SARC           0
    PRAD           0
    PCPG           0
    PAAD           0
      OV           0
    THYM           0
     LGG           0
    LIHC           0
    KIRC           0
   KIPAN           0
    HNSC           0
  GBMLGG           0
    CESC           0
    BRCA           0
    LAML           0
    UCEC           1
    LUAD           2
    LUSC           2
    COAD           2
    STES           2
    BLCA           2
    DLBC           3
    SKCM           4

[클러스터별 암종 구성]

Cluster 0: ['ACC', 'BRCA', 'CESC', 'GBMLGG', 'HNSC', 'KIPAN', 'KIRC', 'LAML', 'LGG', 'LIHC', 'OV', 'PAAD', 'PCPG', 'PRAD', 'SARC', 'TGCT', 'THCA', 'THYM']

Cluster 1: ['UCEC']

Cluster 2: ['BLCA', 'COAD', 'LUAD', 'LUSC', 'STES']

Cluster 3: ['DLBC']

Cluster 4: ['SKCM']

[클러스터링 분석 완료]


In [5]:
# =============================================================
# [Step 5-1] PyTorch Dataset 구성
# =============================================================

class CancerDataset(Dataset):

    def __init__(self, features, super_labels, sub_labels):

        self.features = torch.tensor(
            features, dtype=torch.float32
        )

        self.super_labels = torch.tensor(
            np.asarray(super_labels),
            dtype=torch.long
        )

        self.sub_labels = torch.tensor(
            np.asarray(sub_labels),
            dtype=torch.long
        )

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):

        return (
            self.features[idx],
            self.super_labels[idx],
            self.sub_labels[idx]
        )


# Step 3에서 이미 분리한 데이터를 그대로 사용
train_dataset = CancerDataset(
    X_train_gene,
    y_super_train,
    y_sub_train
)

val_dataset = CancerDataset(
    X_val_gene,
    y_super_val,
    y_sub_val
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    drop_last=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False
)

print("[Dataset 구성 완료]")
print(f"Train samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")

[Dataset 구성 완료]
Train samples: 4960
Validation samples: 1241


In [6]:
# =============================================================
# [Step 5-2] 디바이스 및 클래스 가중치 설정
# =============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

input_dim = X_train_gene.shape[1]

# Train 데이터만 사용하여 클래스 가중치 계산
sub_classes_array = np.asarray(y_sub_train)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(num_sub_classes),
    y=sub_classes_array
)

class_weights_tensor = torch.tensor(
    class_weights,
    dtype=torch.float32
).to(device)

print(f"\n사용 디바이스: {device}")
print(f"Input dimension: {input_dim}")
print(f"Subclass classes: {num_sub_classes}")
print(f"Super classes: {num_super_classes}")


사용 디바이스: cpu
Input dimension: 4230
Subclass classes: 26
Super classes: 10


In [7]:
# =============================================================
# [Step 5-3] 계층적 멀티태스크 분류 모델
# =============================================================

class HierarchicalCancerClassifier(nn.Module):

    def __init__(
        self,
        input_dim,
        num_super_classes,
        num_sub_classes,
        hidden_dim=256
    ):

        super().__init__()

        # 공통 특징 추출기
        self.shared_encoder = nn.Sequential(

            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.4),

            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        # 상위 그룹 분류기
        self.super_classifier = nn.Linear(
            hidden_dim // 2,
            num_super_classes
        )

        # 세부 암종 분류기
        self.sub_classifier = nn.Linear(
            hidden_dim // 2,
            num_sub_classes
        )

    def forward(self, x):

        features = self.shared_encoder(x)

        super_logits = self.super_classifier(features)
        sub_logits = self.sub_classifier(features)

        return super_logits, sub_logits


model = HierarchicalCancerClassifier(
    input_dim=input_dim,
    num_super_classes=num_super_classes,
    num_sub_classes=num_sub_classes,
    hidden_dim=256
).to(device)

In [8]:
# =============================================================
# [Step 5-4] Loss 및 Optimizer 설정
# =============================================================

criterion_sub = nn.CrossEntropyLoss(
    weight=class_weights_tensor
)

criterion_super = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

# 상위 그룹 보조 손실 가중치
alpha = 0.4

print("\n모델 및 손실 함수 설정 완료!")




모델 및 손실 함수 설정 완료!


In [9]:
# =============================================================
# [Step 5-5] 멀티태스크 학습
# =============================================================

num_epochs = 50

history = {
    "train_loss": [],
    "val_loss": [],
    "val_sub_acc": [],
    "val_sub_f1": [],
    "val_super_acc": []
}

best_val_f1 = -1.0
best_model_state = None

print("\n[모델 학습 시작]")

for epoch in range(num_epochs):

    # ---------------------------------------------------------
    # Training
    # ---------------------------------------------------------

    model.train()

    train_loss_sum = 0.0
    train_sample_count = 0

    for features, super_targets, sub_targets in train_loader:

        features = features.to(device)
        super_targets = super_targets.to(device)
        sub_targets = sub_targets.to(device)

        optimizer.zero_grad()

        super_logits, sub_logits = model(features)

        loss_super = criterion_super(
            super_logits,
            super_targets
        )

        loss_sub = criterion_sub(
            sub_logits,
            sub_targets
        )

        total_loss = loss_sub + alpha * loss_super

        total_loss.backward()

        optimizer.step()

        batch_size = features.size(0)

        train_loss_sum += total_loss.item() * batch_size
        train_sample_count += batch_size

    train_loss = train_loss_sum / train_sample_count

    # ---------------------------------------------------------
    # Validation
    # ---------------------------------------------------------

    model.eval()

    val_loss_sum = 0.0
    val_sample_count = 0

    val_sub_preds = []
    val_sub_targets = []

    val_super_preds = []
    val_super_targets = []

    with torch.no_grad():

        for features, super_targets, sub_targets in val_loader:

            features = features.to(device)
            super_targets = super_targets.to(device)
            sub_targets = sub_targets.to(device)

            super_logits, sub_logits = model(features)

            loss_super = criterion_super(
                super_logits,
                super_targets
            )

            loss_sub = criterion_sub(
                sub_logits,
                sub_targets
            )

            total_loss = loss_sub + alpha * loss_super

            batch_size = features.size(0)

            val_loss_sum += total_loss.item() * batch_size
            val_sample_count += batch_size

            sub_preds = torch.argmax(
                sub_logits,
                dim=1
            )

            super_preds = torch.argmax(
                super_logits,
                dim=1
            )

            val_sub_preds.extend(
                sub_preds.cpu().numpy()
            )

            val_sub_targets.extend(
                sub_targets.cpu().numpy()
            )

            val_super_preds.extend(
                super_preds.cpu().numpy()
            )

            val_super_targets.extend(
                super_targets.cpu().numpy()
            )

    val_loss = val_loss_sum / val_sample_count

    # Subclass 성능
    val_sub_acc = accuracy_score(
        val_sub_targets,
        val_sub_preds
    )

    val_sub_f1 = f1_score(
        val_sub_targets,
        val_sub_preds,
        average="macro",
        zero_division=0
    )

    # Super Class 성능
    val_super_acc = accuracy_score(
        val_super_targets,
        val_super_preds
    )

    # 학습 기록
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_sub_acc"].append(val_sub_acc)
    history["val_sub_f1"].append(val_sub_f1)
    history["val_super_acc"].append(val_super_acc)

    # Macro F1 기준 최적 모델 저장
    if val_sub_f1 > best_val_f1:

        best_val_f1 = val_sub_f1

        best_model_state = {
            k: v.detach().cpu().clone()
            for k, v in model.state_dict().items()
        }

    print(
        f"Epoch [{epoch+1:02d}/{num_epochs}] "
        f"| Train Loss: {train_loss:.4f} "
        f"| Val Loss: {val_loss:.4f} "
        f"| Sub Acc: {val_sub_acc:.4f} "
        f"| Sub Macro F1: {val_sub_f1:.4f} "
        f"| Super Acc: {val_super_acc:.4f}"
    )



[모델 학습 시작]
Epoch [01/50] | Train Loss: 3.7102 | Val Loss: 3.0357 | Sub Acc: 0.2917 | Sub Macro F1: 0.2542 | Super Acc: 0.5044
Epoch [02/50] | Train Loss: 2.4513 | Val Loss: 2.6305 | Sub Acc: 0.3513 | Sub Macro F1: 0.3175 | Super Acc: 0.5705
Epoch [03/50] | Train Loss: 1.4496 | Val Loss: 2.7411 | Sub Acc: 0.3360 | Sub Macro F1: 0.3290 | Super Acc: 0.5649
Epoch [04/50] | Train Loss: 0.7891 | Val Loss: 2.9376 | Sub Acc: 0.3296 | Sub Macro F1: 0.3187 | Super Acc: 0.5471
Epoch [05/50] | Train Loss: 0.4734 | Val Loss: 3.1665 | Sub Acc: 0.3167 | Sub Macro F1: 0.3139 | Super Acc: 0.5536
Epoch [06/50] | Train Loss: 0.3478 | Val Loss: 3.3965 | Sub Acc: 0.3118 | Sub Macro F1: 0.3104 | Super Acc: 0.5455
Epoch [07/50] | Train Loss: 0.2888 | Val Loss: 3.6484 | Sub Acc: 0.3030 | Sub Macro F1: 0.2956 | Super Acc: 0.5262
Epoch [08/50] | Train Loss: 0.2401 | Val Loss: 3.7275 | Sub Acc: 0.2949 | Sub Macro F1: 0.2925 | Super Acc: 0.5173
Epoch [09/50] | Train Loss: 0.2184 | Val Loss: 3.8779 | Sub Acc: 0.2

In [10]:
# =============================================================
# [Step 5-6] 최적 모델 복원
# =============================================================

if best_model_state is None:
    raise RuntimeError("학습 중 최적 모델이 저장되지 않았습니다.")

model.load_state_dict(best_model_state)
model.to(device)

print("\n[학습 완료]")
print(f"Best Validation Subclass Macro F1: {best_val_f1:.4f}")


[학습 완료]
Best Validation Subclass Macro F1: 0.3290


In [11]:
# =============================================================
# [Step 6-1] Validation 예측 수행
# =============================================================

model.eval()

val_preds_sub = []
val_preds_super = []

val_true_sub = []
val_true_super = []

with torch.no_grad():

    for features, super_targets, sub_targets in val_loader:

        features = features.to(device)

        super_logits, sub_logits = model(features)

        preds_sub = torch.argmax(sub_logits, dim=1)
        preds_super = torch.argmax(super_logits, dim=1)

        val_preds_sub.extend(
            preds_sub.cpu().numpy()
        )

        val_preds_super.extend(
            preds_super.cpu().numpy()
        )

        val_true_sub.extend(
            sub_targets.numpy()
        )

        val_true_super.extend(
            super_targets.numpy()
        )

In [12]:
# =============================================================
# [Step 6-2] 성능 평가
# =============================================================

# Subclass 평가
sub_accuracy = accuracy_score(
    val_true_sub,
    val_preds_sub
)

sub_macro_f1 = f1_score(
    val_true_sub,
    val_preds_sub,
    average="macro",
    zero_division=0
)

sub_weighted_f1 = f1_score(
    val_true_sub,
    val_preds_sub,
    average="weighted",
    zero_division=0
)

# Super Class 평가
super_accuracy = accuracy_score(
    val_true_super,
    val_preds_super
)

super_macro_f1 = f1_score(
    val_true_super,
    val_preds_super,
    average="macro",
    zero_division=0
)

print("\n========== Validation 성능 평가 ==========")

print(
    f"Subclass Accuracy: {sub_accuracy:.4f}"
)

print(
    f"Subclass Macro F1: {sub_macro_f1:.4f}"
)

print(
    f"Subclass Weighted F1: {sub_weighted_f1:.4f}"
)

print(
    f"Super Class Accuracy: {super_accuracy:.4f}"
)

print(
    f"Super Class Macro F1: {super_macro_f1:.4f}"
)


========== Validation 성능 평가 ==========
Subclass Accuracy: 0.3360
Subclass Macro F1: 0.3290
Subclass Weighted F1: 0.3351
Super Class Accuracy: 0.5649
Super Class Macro F1: 0.4978


In [13]:
# =============================================================
# [Step 6-3] 예측 결과 DataFrame 생성
# =============================================================

val_results_df = pd.DataFrame({

    "true_sub_label": val_true_sub,
    "pred_sub_label": val_preds_sub,

    "true_super_label": val_true_super,
    "pred_super_label": val_preds_super

})

# 원래 암종 이름 복원
val_results_df["true_sub_name"] = (
    sub_encoder.inverse_transform(
        val_results_df["true_sub_label"]
    )
)

val_results_df["pred_sub_name"] = (
    sub_encoder.inverse_transform(
        val_results_df["pred_sub_label"]
    )
)

# 상위 그룹 이름 복원
val_results_df["true_super_name"] = (
    super_encoder.inverse_transform(
        val_results_df["true_super_label"]
    )
)

val_results_df["pred_super_name"] = (
    super_encoder.inverse_transform(
        val_results_df["pred_super_label"]
    )
)

# 오분류 여부
val_results_df["is_error"] = (
    val_results_df["true_sub_label"]
    != val_results_df["pred_sub_label"]
)

# 상위 그룹 오분류 여부
val_results_df["is_super_error"] = (
    val_results_df["true_super_label"]
    != val_results_df["pred_super_label"]
)

print("\n[Validation 예측 결과 미리보기]")
print(val_results_df.head())



[Validation 예측 결과 미리보기]
   true_sub_label  pred_sub_label  true_super_label  pred_super_label  \
0              22              23                 8                 1   
1               8               8                 6                 6   
2              21              21                 3                 3   
3              25              25                 2                 2   
4              14              14                 7                 7   

  true_sub_name pred_sub_name            true_super_name  \
0          TGCT          THCA   Male_Genitourinary_Group   
1         KIPAN         KIPAN               Kidney_Group   
2          STES          STES         GI_Digestive_Group   
3          UCEC          UCEC  Female_Reproductive_Group   
4          LUSC          LUSC                 Lung_Group   

             pred_super_name  is_error  is_super_error  
0       Endocrine_Rare_Group      True            True  
1               Kidney_Group     False           False  
2   

In [14]:
# =============================================================
# [Step 6-4] 암종별 Classification Report
# =============================================================

print("\n========== Subclass Classification Report ==========")

print(
    classification_report(
        val_true_sub,
        val_preds_sub,
        labels=np.arange(num_sub_classes),
        target_names=sub_encoder.classes_,
        zero_division=0
    )
)

print("\n========== Super Class Classification Report ==========")

print(
    classification_report(
        val_true_super,
        val_preds_super,
        labels=np.arange(num_super_classes),
        target_names=super_encoder.classes_,
        zero_division=0
    )
)


========== Subclass Classification Report ==========
              precision    recall  f1-score   support

         ACC       0.73      0.79      0.76        14
        BLCA       0.25      0.14      0.18        21
        BRCA       0.62      0.38      0.47       157
        CESC       0.11      0.26      0.16        31
        COAD       0.57      0.51      0.54        45
        DLBC       0.67      0.29      0.40         7
      GBMLGG       0.28      0.12      0.17        92
        HNSC       0.32      0.13      0.19        45
       KIPAN       0.18      0.13      0.15       103
        KIRC       0.24      0.37      0.29        67
        LAML       0.40      0.59      0.48        32
         LGG       0.38      0.65      0.48        46
        LIHC       0.20      0.19      0.20        31
        LUAD       0.23      0.14      0.17        37
        LUSC       0.32      0.33      0.33        36
          OV       0.14      0.16      0.15        51
        PAAD       0.33    

In [15]:
# =============================================================
# [Step 6-5] 오분류 환자군 추출
# =============================================================

error_samples_df = val_results_df[
    val_results_df["is_error"]
].copy()

total_samples = len(val_results_df)
error_count = len(error_samples_df)

error_rate = (
    error_count / total_samples * 100
)

print("\n========== 오분류 분석 ==========")

print(f"Validation 전체 샘플: {total_samples}")
print(f"오분류 샘플: {error_count}")
print(f"오분류율: {error_rate:.2f}%")



========== 오분류 분석 ==========
Validation 전체 샘플: 1241
오분류 샘플: 824
오분류율: 66.40%


In [16]:
# =============================================================
# [Step 6-6] 암종별 오분류율 분석
# =============================================================

subclass_error_analysis = (
    val_results_df
    .groupby("true_sub_name")
    .agg(
        Total_Count=("is_error", "size"),
        Error_Count=("is_error", "sum")
    )
)

subclass_error_analysis["Error_Rate"] = (
    subclass_error_analysis["Error_Count"]
    / subclass_error_analysis["Total_Count"]
)

subclass_error_analysis["Accuracy"] = (
    1 - subclass_error_analysis["Error_Rate"]
)

subclass_error_analysis = (
    subclass_error_analysis
    .sort_values("Error_Rate", ascending=False)
)

print("\n[암종별 오분류율]")
print(
    subclass_error_analysis.to_string()
)



[암종별 오분류율]
               Total_Count  Error_Count  Error_Rate  Accuracy
true_sub_name                                                
GBMLGG                  92           81    0.880435  0.119565
KIPAN                  103           90    0.873786  0.126214
HNSC                    45           39    0.866667  0.133333
LUAD                    37           32    0.864865  0.135135
BLCA                    21           18    0.857143  0.142857
OV                      51           43    0.843137  0.156863
TGCT                    25           21    0.840000  0.160000
LIHC                    31           25    0.806452  0.193548
SARC                    40           32    0.800000  0.200000
PAAD                    24           19    0.791667  0.208333
PRAD                    53           41    0.773585  0.226415
CESC                    31           23    0.741935  0.258065
DLBC                     7            5    0.714286  0.285714
LUSC                    36           24    0.666667  0.333

In [17]:
# =============================================================
# [Step 6-7] 암종 간 오분류 패턴 분석
# =============================================================

confusion_pairs = (
    error_samples_df
    .groupby(
        ["true_sub_name", "pred_sub_name"]
    )
    .size()
    .reset_index(name="confusion_count")
    .sort_values(
        "confusion_count",
        ascending=False
    )
)

print("\n[가장 빈번한 암종 간 오분류 패턴 Top 10]")

print(
    confusion_pairs.head(10).to_string(index=False)
)


[가장 빈번한 암종 간 오분류 패턴 Top 10]
true_sub_name pred_sub_name  confusion_count
        KIPAN          KIRC               63
       GBMLGG           LGG               45
         KIRC         KIPAN               35
         BRCA            OV               17
         BRCA          CESC               12
          LGG        GBMLGG               11
         BRCA          UCEC               11
         HNSC          STES               10
         STES          CESC               10
         HNSC          UCEC                9


In [18]:
# =============================================================
# [Step 6-8] 상위 그룹 간 오분류 분석
# =============================================================

super_error_pairs = (
    val_results_df[
        val_results_df["is_super_error"]
    ]
    .groupby(
        ["true_super_name", "pred_super_name"]
    )
    .size()
    .reset_index(name="confusion_count")
    .sort_values(
        "confusion_count",
        ascending=False
    )
)

print("\n[상위 그룹 간 오분류 패턴]")

print(
    super_error_pairs.to_string(index=False)
)



[상위 그룹 간 오분류 패턴]
          true_super_name           pred_super_name  confusion_count
       GI_Digestive_Group Female_Reproductive_Group               48
Female_Reproductive_Group        GI_Digestive_Group               32
       Skin_Sarcoma_Group Female_Reproductive_Group               30
 Male_Genitourinary_Group Female_Reproductive_Group               24
Female_Reproductive_Group      Endocrine_Rare_Group               21
               Lung_Group        GI_Digestive_Group               20
          CNS_Brain_Group Female_Reproductive_Group               18
 Male_Genitourinary_Group      Endocrine_Rare_Group               17
          Head_Neck_Group Female_Reproductive_Group               17
Female_Reproductive_Group              Kidney_Group               17
               Lung_Group Female_Reproductive_Group               16
 Male_Genitourinary_Group        GI_Digestive_Group               15
 Male_Genitourinary_Group              Kidney_Group               15
             Kid

In [19]:
# =============================================================
# [Step 6-9] 결과 저장
# =============================================================

val_results_df.to_csv(
    "validation_predictions.csv",
    index=False
)

subclass_error_analysis.to_csv(
    "subclass_error_analysis.csv"
)

confusion_pairs.to_csv(
    "subclass_confusion_pairs.csv",
    index=False
)

super_error_pairs.to_csv(
    "superclass_confusion_pairs.csv",
    index=False
)

print("\nValidation 분석 결과 저장 완료!")


Validation 분석 결과 저장 완료!


In [20]:
# =============================================================
# [Step 7-1] 예측 함수 및 디코딩 방식 비교 (Validation)
# =============================================================
import torch.nn.functional as F

# 세부 암종 -> 상위 그룹 매핑 (학습 데이터 기준)
sub_to_super = (
    train.groupby("sub_label")["super_label"].first()
    .reindex(range(num_sub_classes)).to_numpy()
)


@torch.no_grad()
def predict_proba(model, features, batch_size=256):
    model.eval()
    sub_list, super_list = [], []
    for i in range(0, len(features), batch_size):
        xb = torch.tensor(features[i:i + batch_size], dtype=torch.float32).to(device)
        super_logits, sub_logits = model(xb)
        sub_list.append(F.softmax(sub_logits, dim=1).cpu().numpy())
        super_list.append(F.softmax(super_logits, dim=1).cpu().numpy())
    return np.vstack(sub_list), np.vstack(super_list)


def hierarchical_decode(P_sub, P_super, alpha=1.0):
    """세부 암종 확률에 '그 암종이 속한 상위 그룹의 확률'을 곱해 두 헤드를 일치시킨다"""
    return P_sub * (P_super[:, sub_to_super] ** alpha)


P_sub_val, P_super_val = predict_proba(model, X_val_gene)
y_val = np.asarray(y_sub_val)

for name, scores in [
    ("sub head만",          P_sub_val),
    ("계층 디코딩 (a=1.0)", hierarchical_decode(P_sub_val, P_super_val, 1.0)),
    ("계층 디코딩 (a=0.5)", hierarchical_decode(P_sub_val, P_super_val, 0.5)),
]:
    pred = scores.argmax(1)
    print(f"{name:18s} Acc {accuracy_score(y_val, pred):.4f} | "
        f"Macro F1 {f1_score(y_val, pred, average='macro', zero_division=0):.4f}")

sub head만          Acc 0.3360 | Macro F1 0.3290
계층 디코딩 (a=1.0)     Acc 0.3320 | Macro F1 0.3122
계층 디코딩 (a=0.5)     Acc 0.3360 | Macro F1 0.3173


In [21]:
# =============================================================
# [Step 7-2] Test 예측 및 제출 파일 저장
# =============================================================
DECODE = "sub"        # "sub" 또는 "hier". 7-1에서 Macro F1이 높은 쪽으로 선택
HIER_ALPHA = 1.0      # DECODE="hier"일 때만 사용 (7-1에서 고른 a 값)

# 입력 정합성 확인
assert X_test_gene.shape[1] == input_dim, "test 입력 차원이 학습과 다름"
assert len(test) == X_test_gene.shape[0]
assert test["ID"].is_unique

P_sub_test, P_super_test = predict_proba(model, X_test_gene)
scores_test = (P_sub_test if DECODE == "sub"
            else hierarchical_decode(P_sub_test, P_super_test, HIER_ALPHA))

pred_sub_test = scores_test.argmax(1)
pred_super_test = P_super_test.argmax(1)

pred_sub_names = sub_encoder.inverse_transform(pred_sub_test)
pred_super_names = super_encoder.inverse_transform(pred_super_test)

# 제출 파일 (ID, SUBCLASS)
submission_nn = pd.DataFrame({"ID": test["ID"].values, "SUBCLASS": pred_sub_names})
assert submission_nn["SUBCLASS"].notna().all()
submission_nn.to_csv("submission_nn.csv", index=False)

# 분석용 상세 파일 (예측 암종, 상위 그룹, 확신도)
detail_df = pd.DataFrame({
    "ID": test["ID"].values,
    "pred_sub_name": pred_sub_names,
    "pred_super_name": pred_super_names,
    "max_prob": scores_test.max(1) / scores_test.sum(1),
})
detail_df.to_csv("test_predictions_detail.csv", index=False)
np.save("P_sub_test_nn.npy", P_sub_test)
np.save("P_super_test_nn.npy", P_super_test)

# 점검 1: 예측 비율을 train 비율과 비교
check = pd.DataFrame({
    "train(%)": train["SUBCLASS"].value_counts(normalize=True) * 100,
    "test예측(%)": submission_nn["SUBCLASS"].value_counts(normalize=True) * 100,
}).round(1).fillna(0)
check["배수"] = (check["test예측(%)"] / check["train(%)"]).round(2)
print(check.sort_values("배수"))

# 점검 2: 세부 암종 예측이 상위 그룹 예측과 일치하는 비율
consistent = (sub_to_super[pred_sub_test] == pred_super_test).mean()
print(f"\nsub 예측과 super 예측이 일치하는 비율: {consistent:.1%}")
print(submission_nn.head(), submission_nn.shape)

          train(%)  test예측(%)    배수
SUBCLASS                           
DLBC           0.6        0.2  0.33
PRAD           4.3        1.5  0.35
GBMLGG         7.4        3.1  0.42
HNSC           3.6        1.6  0.44
LUAD           3.0        1.5  0.50
BRCA          12.7        6.8  0.54
THCA           5.2        2.9  0.56
BLCA           1.7        1.1  0.65
LUSC           2.9        2.0  0.69
KIPAN          8.3        6.3  0.76
OV             4.1        3.1  0.76
SKCM           4.5        3.5  0.78
PAAD           1.9        1.5  0.79
LIHC           2.5        2.0  0.80
KIRC           5.4        4.4  0.81
LGG            3.7        3.4  0.92
LAML           2.5        2.6  1.04
TGCT           2.0        2.1  1.05
SARC           3.2        3.7  1.16
ACC            1.2        1.4  1.17
PCPG           2.4        2.8  1.17
THYM           1.6        2.2  1.38
CESC           2.5        4.7  1.88
UCEC           3.2        7.5  2.34
STES           6.1       17.1  2.80
COAD           3.6       10.